# M5.0 · The negative heterogeneous experiment — naive CPU offload

**Learning objective:** prove *physically* why moving the 8B text encoder from GPU to CPU PyTorch can **lose** throughput: the GPU idles during multi-minute CPU encodes. This negative result is taught with real work (bounded pilots on your host), never only with a toy simulator — and the simulator, where used, is labelled `SIMULATED`.

**Predict first:** the GPU text encoder encodes in ~0.55 s warm. How long will the same encode take on 16 CPU cores in PyTorch? Write your estimate (the frozen campaign says two orders of magnitude — verify on your host).

Reference lesson: [M05 lesson](../course/M05_parallel_heterogeneous/LESSON.md) · [中文](./05_cpu_offload_negative.zh-CN.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse explicitly selected physical runs (ids shown + classified)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="05")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

from lab import student_provenance as prov

def _origin_stamp(category: str) -> str:
    """Stamp text for a provenance category; reuse is NEVER labelled as
    measured-today (fail-closed student provenance contract)."""
    return prov.stamp_text(category)

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse earlier physical runs to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing explicitly selected physical run(s) "
          "instead of launching new ones (metered-GPU mode; provenance shown)")
    return False

## 1 · The frozen reference case (GPU vs PyTorch-CPU encoder)
The committed cleanroom study measured the same frozen prompts on the same host: GPU PyTorch vs CPU PyTorch text encoder. Inspect the per-prompt warm medians and the ratio.

In [ ]:
from lab import notebook_support as ns
import statistics

enc = ns.read_json("runs/m51-20261105-cleanroom-01/performance/encoder_ab.json")
g = enc["arms"]["G_PyTorch_GPU"]["prompts"]
c0 = enc["arms"]["C0_PyTorch_CPU"]["prompts"]
for pid in ("p1-zh-scene", "p2-en-scene"):
    gm = g[pid]["warm_median_s"]
    cm = c0[pid]["warm_median_s"]
    print(f"{pid:14s} GPU {gm:7.3f}s   CPU {cm:8.2f}s   CPU/GPU = {cm/gm:7.1f}x")
_stamp(HISTORICAL, "frozen cleanroom campaign: PyTorch CPU TE is ~2 orders "
      "of magnitude slower than GPU TE on the same host")

## 2 · Your own bounded physical pilot (GPU modes)
One isolated probe per device, one prompt, one warm repeat — a **bounded pilot**, explicitly not a full-protocol claim. The CPU arm can take minutes; that slowness *is* the measurement.

In [ ]:
def run_te_pilots():
    gpu = session.run_m51_baseline("cuda", repeats=1)
    cpu = session.run_m51_baseline("cpu", repeats=1)
    return gpu, cpu

pilots = gpu_cell(run_te_pilots) if want_own_arms() else None
own = None
if pilots is not None:
    gpr = json.loads((pilots[0] / "probe_cuda.json").read_text())
    cpr = json.loads((pilots[1] / "probe_cpu.json").read_text())
    gp = next(iter(gpr["prompts"].values()))
    cp = next(iter(cpr["prompts"].values()))
    g_warm = gp["warm_samples_s"] if isinstance(gp, dict) else [r["encode_s"] for r in gp]
    c_warm = cp["warm_samples_s"] if isinstance(cp, dict) else [r["encode_s"] for r in cp]
    own = {"gpu_warm_s": g_warm[-1],
           "cpu_warm_s": c_warm[-1],
           "cpu_rss_mb": cpr.get("rss_peak_mb"),
           "gpu_rss_mb": gpr.get("rss_peak_mb"),
           "provenance": "CURRENT_SESSION_PHYSICAL",
           "run_ids": [pilots[0].name, pilots[1].name]}
    _stamp(MEASURED_TODAY,
           f"pilot: GPU TE {own['gpu_warm_s']:.2f}s vs CPU TE "
           f"{own['cpu_warm_s']:.1f}s -> {own['cpu_warm_s']/own['gpu_warm_s']:.0f}x "
           f"slower (n=1, bounded pilot, single prompt) run_ids={own['run_ids']}")
else:
    ref = enc["arms"]["G_PyTorch_GPU"]["prompts"]["p1-zh-scene"]["warm_median_s"]
    refc = enc["arms"]["C0_PyTorch_CPU"]["prompts"]["p1-zh-scene"]["warm_median_s"]
    own = {"gpu_warm_s": ref, "cpu_warm_s": refc, "cpu_rss_mb": None,
           "gpu_rss_mb": None, "provenance": "COMMITTED_PHYSICAL_REFERENCE",
           "run_ids": ["m51-20261105-cleanroom-01"]}
    _stamp(_origin_stamp("COMMITTED_PHYSICAL_REFERENCE"),
           "no own pilot executed; pilot numbers below are the committed "
           "cleanroom reference (explicit source, NOT your measurement)")

## 3 · The pipeline view — where the GPU sits idle
H1 runs the CPU encoder inside ComfyUI (the integrated arm); H2 runs four requests sequentially (GPU TE) vs pipelined (CPU TE overlapping GPU denoise). Evidence selection is EXPLICIT: the committed qualification-campaign records are consumed by exact run id (never a newest-file glob), and a fresh student clone never mistakes them for its own measurement. GPU_RESEARCH students may run the full H2 themselves.

In [ ]:
# Explicit evidence ids — no glob, no "newest file wins".
H1_REFERENCE_RUN = "q1-075553-H1-cpu-te"
H2_REFERENCE_RUN = "q1-075553-H2-pipeline"
RUN_OWN_H2 = False  # True + GPU_RESEARCH: execute the full H2 pipeline yourself

def load_module_record(run_id, fname, why):
    session.note_consumed(run_id, why=why)
    return json.loads((ROOT / "runs" / run_id / fname).read_text())

h1 = load_module_record(H1_REFERENCE_RUN, "H1_RESULTS.json",
                        "committed campaign H1 (explicit run id)")
for tag, row in h1["integrated"].items():
    print(f"H1 integrated {tag:9s} total={row.get('total_s')}s "
          f"te={row.get('text_encode_s')}s vram_peak={row.get('vram_peak_mb')}MB")
_stamp(_origin_stamp(session.consumed[-1]["category"]),
       f"H1 record runs/{H1_REFERENCE_RUN}/H1_RESULTS.json")

h2 = None
h2_origin = None
if RUN_OWN_H2 and session.mode == "GPU_RESEARCH" and want_own_arms():
    bundle = session.run("H2", profile="research")
    h2 = json.loads((ROOT / "runs" / bundle.run_id / "H2_RESULTS.json").read_text())
    h2_origin = "CURRENT_SESSION_PHYSICAL"
    _stamp(MEASURED_TODAY, f"own H2 pipeline run_id={bundle.run_id}")
else:
    h2 = load_module_record(H2_REFERENCE_RUN, "H2_RESULTS.json",
                            "committed campaign H2 (explicit run id)")
    h2_origin = session.consumed[-1]["category"]
    for k, v in h2.items():
        if isinstance(v, (int, float)):
            print(f"H2 {k}: {v}")
    _stamp(_origin_stamp(h2_origin),
           f"H2 record runs/{H2_REFERENCE_RUN}/H2_RESULTS.json — classroom "
           "scope keeps the H2 comparison reference-only (no fresh pipeline "
           "claim is made from it)")

## 4 · The scheduler simulator — clearly labelled
The Gantt-style simulation below illustrates *why* overlap helps once the CPU encoder is fast enough (M5.1's OpenVINO). Its numbers are **SIMULATED**, never measured.

In [ ]:
_stamp(ESTIMATE, "SIMULATED pipeline schedule - illustrative only "
      f"(inputs: {own['provenance']})")

gpu_t = own["gpu_warm_s"]    # TE on GPU (s)
dit_t = 12.0                 # DiT+VAE per request (s, order of M1/B0)
cpu_t = own["cpu_warm_s"]    # TE on CPU (s)
seq_gpu = 4 * (gpu_t + dit_t)                  # 4 requests, GPU TE
seq_cpu = 4 * (cpu_t + dit_t)                  # 4 requests, naive CPU offload
pipelined = max(4 * cpu_t, gpu_t + 4 * dit_t)  # CPU TE overlapped with GPU work
print(f"SIMULATED 4-request wall: GPU-TE {seq_gpu:.1f}s | naive CPU offload "
      f"{seq_cpu:.1f}s | overlapped {pipelined:.1f}s")
print("naive offload LOSES unless CPU encode overlaps GPU denoise")

## 5 · Your turn
1. With your pilot numbers, how many CPU workers would the overlapped design need to hide the CPU encode fully?
2. Why does 'CPU offload' still win on **VRAM** even when it loses on latency? When is that the right trade?
3. Name one reason your host's CPU/GPU ratio may differ from the frozen campaign's 2 orders of magnitude.

In [ ]:
m50_result = {
    "experiment_id": "M50",
    "session_id": session.session_id,
    "course_mode": session.mode,
    "run_ids": list(own.get("run_ids", [])),
    "provenance": own["provenance"],
    "execution_status": ("MEASURED"
                         if own["provenance"] == "CURRENT_SESSION_PHYSICAL"
                         else "REUSED"),
    "gpu": {"arch": (session.admission.detail.get("gfx_arch")
                    if session.admission and session.admission.detail else None),
            "model": (session.admission.detail.get("device_name")
                      if session.admission and session.admission.detail else None)},
    "source_head": session.source.get("head"),
    "source_dirty": session.source.get("dirty"),
    "workload": {"experiment": "te-pilot", "prompt": "p1-zh-scene",
                 "repeats": 1, "h2_record": H2_REFERENCE_RUN},
    "populations": {"warmup_count": 1, "measured_count": 1,
                    "cold_warm": "warm pilot sample"},
    "measurements": {"gpu_te_warm_s": own["gpu_warm_s"],
                     "cpu_te_warm_s": own["cpu_warm_s"],
                     "cpu_rss_mb": own.get("cpu_rss_mb")},
    "throughput": None,
    "memory": {"cpu_rss_mb": own.get("cpu_rss_mb"), "gpu_vram_peak_mb": None},
    "numerical_status": "NOT_ASSESSED",
    "human_quality_status": "NOT_ASSESSED",
    "classification": ("BOUNDED_PHYSICAL_COMPARISON"
                       if own["provenance"] == "CURRENT_SESSION_PHYSICAL"
                       else "REFERENCE_ONLY"),
    "ac_pair": None,
    "result_status": "PASS",
    "source_paths": [f"runs/{r}" for r in own.get("run_ids", [])],
    "reproduce_command": ("python -u lab/m51_baseline.py --device cuda|cpu "
                          "--out <fresh-tag> --repeats 1"),
}
prov.validate_or_raise(m50_result, session_id=session.session_id)
print("[STUDENT RESULT VALIDATED]",
      m50_result["experiment_id"], m50_result["provenance"],
      m50_result["classification"])

summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "session_id": session.session_id,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "physical_experiment_status": ("MEASURED"
                                   if own["provenance"] == "CURRENT_SESSION_PHYSICAL"
                                   else "REFERENCE_ONLY"),
    "student_result": m50_result,
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## Acceptance criteria

- [ ] You can state which numbers are **measured on this GPU today**, which are
      **historical reference values**, and which are **estimates/simulations**.
- [ ] Every physical run you produced has a visible `run_id` under `runs/`.
- [ ] You did not present a smoke/classroom population as a formal result.
- [ ] You can explain one limitation of each measurement you reported.

## Reproduce & contribute

```bash
python -m lab doctor                                  # admission evidence -> runs/doctor.json
python -m lab run --experiment H1 --date <your-tag>  # full research protocol
python -m lab report --scope student --kind all       # aggregate YOUR runs only
```

Commit your `runs/<run-id>/*.json` records (never `evidence/`) and open a PR
describing: hypothesis → protocol → numbers → limitations. Negative results
are welcome — a measured regression is a scientific outcome, not a failure.